In [ ]:
# Notebook 03 – XGBoost & Isolation Forest
# Objective: Train both a supervised (XGBoost) and an unsupervised (Isolation Forest) model
#            to detect credit card fraud, and evaluate their performance on imbalanced data.

# === Import Required Libraries ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    precision_recall_curve
)

from sklearn.ensemble import IsolationForest
import xgboost as xgb

# === Load and Prepare Dataset ===
df = pd.read_csv('../data/raw/creditcard.csv')

# Scale continuous features
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled'] = StandardScaler().fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

# Split features and target
X = df.drop(columns='Class')
y = df['Class']

# Stratified train-test split to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# === XGBoost Classifier with Class Weighting ===

# Compute class imbalance ratio for scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Initialize and train XGBoost classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

# Predictions and probabilities
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate XGBoost model
print("XGBoost Classification Report")
print(classification_report(y_test, y_pred_xgb))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_xgb):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_xgb):.4f}")

# === Isolation Forest (Unsupervised Learning) ===

# Train Isolation Forest model (unsupervised anomaly detection)
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.0017,  # Approximate fraud ratio
    random_state=42
)
iso_forest.fit(X_train)

# Predict outliers: -1 = anomaly → mapped to fraud (1), 1 = normal → mapped to legit (0)
y_pred_if = iso_forest.predict(X_test)
y_pred_if = np.where(y_pred_if == -1, 1, 0)

# Evaluate Isolation Forest predictions
print("Isolation Forest Classification Report")
print(classification_report(y_test, y_pred_if))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_if))

# === Precision–Recall Curve for XGBoost ===
precision, recall, _ = precision_recall_curve(y_test, y_proba_xgb)

plt.figure(figsize=(6, 4))
plt.plot(recall, precision, label='XGBoost', color='blue')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–Recall Curve – XGBoost')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
